In [1]:
%load_ext autoreload
%autoreload 2
from trade.datamanager import (
    DividendDataManager,
    SpotDataManager,
    OptionSpotDataManager,
    VolDataManager,
    RatesDataManager,
    BaseDataManager,
    ForwardDataManager,
    GreekDataManager,
    assert_synchronized_model,
    get_option_theoretical_price,
    calculate_scenarios,
)

from trade.datamanager._enums import (
    OptionSpotEndpointSource,
    SeriesId,
    OptionPricingModel,
    VolatilityModel,
    RealTimeFallbackOption,
    GreekType,
    ModelPrice,
)
from trade.optionlib.config.types import DivType
from trade.helpers.helper_types import SingletonMetaClass
from trade.datamanager.vars import get_loaded_names, TS
from trade.datamanager.utils.model import LoadRequest, _load_model_data_timeseries
from trade.datamanager.utils.logging import (
    change_datamanager_factor_loggers_level,
    change_datamanager_utils_logging_level,
    change_logging_in_all_datamanager_loggers,
    change_all_optionlib_loggers_level,
)

# change_datamanager_factor_loggers_level("CRITICAL")
# change_datamanager_utils_logging_level("CRITICAL")
# change_logging_in_all_datamanager_loggers("CRITICAL")
change_all_optionlib_loggers_level("CRITICAL")
get_loaded_names()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/Users/chiemelienwanisobi/miniconda3/envs/openbb_new_use/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


2026-02-13 13:06:17 trade.helpers.Logging INFO: Logging Root Directory: /Users/chiemelienwanisobi/cloned_repos/QuantTools/logs
2026-02-13 13:06:17 [test] trade.helpers.clear_cache INFO: No expired caches to delete on 2026-02-13.
2026-02-13 13:06:20 [test] dbase.DataAPI.ThetaData.proxy INFO: Refreshed proxy URL: http://54.205.248.219:5500/thetadata
2026-02-13 13:06:20 [test] dbase.DataAPI.ThetaData.proxy INFO: Using Proxy URL: http://54.205.248.219:5500/thetadata
2026-02-13 13:06:20 [test] dbase.DataAPI.ThetaData INFO: Using V2 of the ThetaData API
Fetching rates data from yfinance directly during market hours
YF.download() has changed argument auto_adjust default to True


set()

In [2]:
BaseDataManager.clear_all_caches()

2026-02-13 13:06:24 [test] trade.datamanager.base INFO: Clearing cache for DividendDataManager (CACHE_NAME='dividend_data_manager')
2026-02-13 13:06:24 [test] trade.datamanager.base INFO: Clearing cache for RatesDataManager (CACHE_NAME='rates_data_manager')
2026-02-13 13:06:24 [test] trade.datamanager.base INFO: Clearing cache for ForwardDataManager (CACHE_NAME='forward_data_manager')
2026-02-13 13:06:24 [test] trade.datamanager.base INFO: Clearing cache for OptionSpotDataManager (CACHE_NAME='option_spot_manager')
2026-02-13 13:06:24 [test] trade.datamanager.base INFO: Clearing cache for SpotDataManager (CACHE_NAME='spot_data_manager')
2026-02-13 13:06:24 [test] trade.datamanager.base INFO: Clearing cache for VolDataManager (CACHE_NAME='vol_data_manager_cache')
2026-02-13 13:06:24 [test] trade.datamanager.base INFO: Clearing cache for GreekDataManager (CACHE_NAME='greek_datamanager_cache')


In [3]:
## Vars
div = DivType.CONTINUOUS
undo_adjust = True
endpoint_source = OptionSpotEndpointSource.EOD
series_id = SeriesId.HIST
market_model = OptionPricingModel.BSM
vol_model = VolatilityModel.MARKET
model_price = ModelPrice.ASK

symbol = "SBUX"
expiration = "2026-09-18"
right = "C"
strike = 100.0
ts_start = "2025-01-01"
ts_end = "2026-01-28"

In [4]:
request = LoadRequest(
    symbol=symbol,
    # start_date=ts_start,
    # end_date=ts_end,
    as_of=ts_end,
    expiration=expiration,
    strike=strike,
    right=right,
    series_id=SeriesId.HIST,
    dividend_type=DivType.DISCRETE,
    endpoint_source=OptionSpotEndpointSource.EOD,
    vol_model=VolatilityModel.MARKET,
    market_model=OptionPricingModel.BINOMIAL,
    model_price=ModelPrice.ASK,
    load_spot=True,
    load_dividend=True,
    load_forward=True,
    load_option_spot=True,
    load_vol=True,
    load_greek=True,
    load_rates=True,
    undo_adjust=True,
    # rt=True,
)
data_packet = _load_model_data_timeseries(request)


2026-02-13 13:06:24 [test] trade.datamanager.vars INFO: Loading timeseries for SBUX...
2026-02-13 13:06:25 [test] trade.datamanager.utils INFO: Cutting off today's data for key: SBUX to avoid saving partial day data.
2026-02-13 13:06:25 [test] trade.datamanager.utils INFO: Caching timeseries data structure for key: SBUX with date range 2017-01-03 to 2026-02-12, missing dates within range: []
2026-02-13 13:06:25 [test] trade.datamanager.market_data INFO: Loaded spot data for symbol SBUX into cache.
2026-02-13 13:06:25 [test] trade.datamanager.utils INFO: Cutting off today's data for key: SBUX to avoid saving partial day data.
2026-02-13 13:06:25 [test] trade.datamanager.utils INFO: Caching timeseries data structure for key: SBUX with date range 2017-01-03 to 2026-02-12, missing dates within range: []
2026-02-13 13:06:25 [test] trade.datamanager.market_data INFO: Loaded chain spot data for symbol SBUX into cache.
2026-02-13 13:06:25 [test] trade.datamanager.market_data_helpers.dividends 

In [5]:
data_packet.greek.fallback_option


<RealTimeFallbackOption.USE_LAST_AVAILABLE: 'use_last_available'>

In [6]:
request.end_date

Timestamp('2026-01-28 00:00:00')

In [7]:
bsm = calculate_scenarios(
    symbol=symbol,
    expiration=expiration,
    right=right,
    strike=strike,
    return_pnl_in_pct=True,
    return_pnl=True,
    rt=True
)


2026-02-13 13:06:34 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:06:34 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:06:34 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:06:34 [test] trade.datamanager.utils INFO: Converting old cache data structure to new for key: symbol:^IRX|interval:eod|artifact_type:rates|series_id:hist|fn_interval:1D
2026-02-13 13:06:34 [test] trade.datamanager.utils INFO: Cached data date range 2026-01-28 to 2026-01-28 does not cover requested range 2026-02-13 to 2026-02-13.
2026-02-13 13:06:34 [test] trade.datamanager.utils INFO: Cache partially covers requested date range for timeseries data structure. Key: symbol:^IRX|interval:eod|artifact_type:rates|series_id:hist|fn_interval:1D. Fetching missing dates within range: []
2026-02-13 13:06:34 [test] trade.datamanager.rates INFO: Cache partially covers requested date range for risk-free rate times

In [8]:
RatesDataManager().rt().timeseries

2026-02-13 13:06:36 [test] trade.datamanager.utils INFO: Converting old cache data structure to new for key: symbol:^IRX|interval:eod|artifact_type:rates|series_id:hist|fn_interval:1D
2026-02-13 13:06:36 [test] trade.datamanager.utils INFO: Cached data date range 2026-01-28 to 2026-01-28 does not cover requested range 2026-02-13 to 2026-02-13.
2026-02-13 13:06:36 [test] trade.datamanager.utils INFO: Cache partially covers requested date range for timeseries data structure. Key: symbol:^IRX|interval:eod|artifact_type:rates|series_id:hist|fn_interval:1D. Fetching missing dates within range: []
2026-02-13 13:06:36 [test] trade.datamanager.rates INFO: Cache partially covers requested date range for risk-free rate timeseries. Key: symbol:^IRX|interval:eod|artifact_type:rates|series_id:hist|fn_interval:1D
2026-02-13 13:06:38 [test] trade.datamanager.utils INFO: Cutting off today's data for key: symbol:^IRX|interval:eod|artifact_type:rates|series_id:hist|fn_interval:1D to avoid saving partial

datetime
2026-02-13    0.03593
Name: annualized, dtype: float64

In [9]:
SpotDataManager("SBUX").rt().timeseries

2026-02-13 13:06:38 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:06:38 [test] trade.datamanager.utils INFO: Cached data date range 2017-01-03 to 2026-02-12 does not cover requested range 2026-02-13 to 2026-02-13.
2026-02-13 13:06:38 [test] trade.datamanager.utils INFO: Cache partially covers requested date range for timeseries data structure. Key: SBUX. Fetching missing dates within range: []
2026-02-13 13:06:38 [test] trade.datamanager.utils INFO: Cutting off today's data for key: SBUX to avoid saving partial day data.
2026-02-13 13:06:38 [test] trade.datamanager.utils INFO: Caching timeseries data structure for key: SBUX with date range 2017-01-03 to 2026-02-12, missing dates within range: []
2026-02-13 13:06:38 [test] trade.datamanager.market_data INFO: Loaded spot data for symbol SBUX into cache.
2026-02-13 13:06:38 [test] trade.datamanager.utils INFO: Cached data date range 2017-01-03 to 2026-02-12 does not cover requested range 2026-02-13 

datetime
2026-02-13    94.635002
Name: close, dtype: object

In [10]:
RatesDataManager()._query_yfinance(
    start_date="2026-02-02",
    end_date="2026-02-02",
    interval="1d",
)

,name,description,daily,annualized
Datetime,,,,
2026-02-02,^IRX,13 WEEK TREASURY BILL,0.000096,0.03578


In [11]:
bsm.grid

,0.90,0.95,1.00,1.05,1.10
-0.02,-0.532749,-0.326299,-0.072656,0.227656,0.572091
-0.01,-0.503521,-0.293032,-0.036298,0.265815,0.610556
0.00,-0.474401,-0.259855,-0.000003,0.303945,0.649025
0.01,-0.445382,-0.226760,0.036233,0.342045,0.687496
0.02,-0.416455,-0.193744,0.072414,0.380116,0.725966


## Batch Load Real Time

In [12]:
option_metas = [
    {
        "symbol": "SBUX",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 100.0,
    },
    {
        "symbol": "SBUX",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 105.0,
    },
    {
        "symbol": "AAPL",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 260.0,
    },
    {
        "symbol": "AAPL",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 265.0,
    },
    {
        "symbol": "AMD",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 280.0,
    },
    {
        "symbol": "AMD",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 290.0,
    },
    {
        "symbol": "META",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 890.0,
    },
    {
        "symbol": "META",
        "expiration": "2026-09-18",
        "right": "C",
        "strike": 900.0,
    }
]

In [13]:
full_data = []
for option_meta in option_metas:
    option_scenarios = calculate_scenarios(
        symbol=option_meta["symbol"],
        expiration=option_meta["expiration"],
        right=option_meta["right"],
        strike=option_meta["strike"],
        return_pnl_in_pct=True,
        return_pnl=True,
        rt=True
    )
    full_data.append(option_scenarios)

2026-02-13 13:06:41 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:06:41 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:06:41 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:06:41 [test] trade.datamanager.utils INFO: Converting old cache data structure to new for key: symbol:^IRX|interval:eod|artifact_type:rates|series_id:hist|fn_interval:1D
2026-02-13 13:06:41 [test] trade.datamanager.utils INFO: Cached data date range 2026-01-28 to 2026-01-28 does not cover requested range 2026-02-13 to 2026-02-13.
2026-02-13 13:06:41 [test] trade.datamanager.utils INFO: Cache partially covers requested date range for timeseries data structure. Key: symbol:^IRX|interval:eod|artifact_type:rates|series_id:hist|fn_interval:1D. Fetching missing dates within range: []
2026-02-13 13:06:41 [test] trade.datamanager.rates INFO: Cache partially covers requested date range for risk-free rate times

In [14]:
timeseries_full_data = []
for option_meta in option_metas:
    request = LoadRequest(
        symbol=option_meta["symbol"],
        start_date=ts_start,
        end_date=ts_end,
        expiration=option_meta["expiration"],
        strike=option_meta["strike"],
        right=option_meta["right"],
        series_id=SeriesId.HIST,
        dividend_type=DivType.DISCRETE,
        endpoint_source=OptionSpotEndpointSource.EOD,
        vol_model=VolatilityModel.MARKET,
        market_model=OptionPricingModel.BINOMIAL,
        model_price=ModelPrice.ASK,
        load_spot=True,
        load_dividend=True,
        load_forward=True,
        load_option_spot=True,
        load_vol=True,
        load_greek=True,
        load_rates=True,
        undo_adjust=True,
        # rt=True,
    )
    data_packet = _load_model_data_timeseries(request)
    timeseries_full_data.append(data_packet)


2026-02-13 13:07:11 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:07:11 [test] trade.datamanager.dividend INFO: Using provided dividend_type: DivType.DISCRETE
2026-02-13 13:07:11 [test] trade.datamanager.dividend INFO: Fetching discrete dividend schedule timeseries for SBUX from 2025-01-01 00:00:00 to 2026-01-28 00:00:00 with maturity 2026-09-18 00:00:00
2026-02-13 13:07:11 [test] trade.datamanager.utils INFO: No cache entry found for key: symbol:SBUX|interval:eod|artifact_type:divs|series_id:hist|current_state:SCHEDULE_TIMESERIES|lookback_years:1|maturity:2026-09-18|method:CONSTANT|undo_adjust:1
2026-02-13 13:07:11 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:07:11 [test] trade.datamanager.utils INFO: No missing dates within cached data range for key: SBUX.
2026-02-13 13:07:11 [test] trade.datamanager.utils INFO: Cache hit for timeseries data structure key: SBUX
2026-02-13 13:07:11 [test] trade.datamanage

In [15]:
timeseries_full_data


[ModelResultPack(symbol='SBUX', strike=100.0, expiration=datetime.datetime(2026, 9, 18, 0, 0), right='C', series_id=<SeriesId.HIST: 'hist'>, dividend_type=<DivType.DISCRETE: 'discrete'>, undo_adjust=True, num_empty=0),
 ModelResultPack(symbol='SBUX', strike=105.0, expiration=datetime.datetime(2026, 9, 18, 0, 0), right='C', series_id=<SeriesId.HIST: 'hist'>, dividend_type=<DivType.DISCRETE: 'discrete'>, undo_adjust=True, num_empty=0),
 ModelResultPack(symbol='AAPL', strike=260.0, expiration=datetime.datetime(2026, 9, 18, 0, 0), right='C', series_id=<SeriesId.HIST: 'hist'>, dividend_type=<DivType.DISCRETE: 'discrete'>, undo_adjust=True, num_empty=0),
 ModelResultPack(symbol='AAPL', strike=265.0, expiration=datetime.datetime(2026, 9, 18, 0, 0), right='C', series_id=<SeriesId.HIST: 'hist'>, dividend_type=<DivType.DISCRETE: 'discrete'>, undo_adjust=True, num_empty=0),
 ModelResultPack(symbol='AMD', strike=280.0, expiration=datetime.datetime(2026, 9, 18, 0, 0), right='C', series_id=<SeriesId

In [16]:
rt_full_data = []
for option_meta in option_metas:
    request = LoadRequest(
        symbol=option_meta["symbol"],

        expiration=option_meta["expiration"],
        strike=option_meta["strike"],
        right=option_meta["right"],
        series_id=SeriesId.HIST,
        dividend_type=DivType.CONTINUOUS,
        endpoint_source=OptionSpotEndpointSource.QUOTE,
        vol_model=VolatilityModel.MARKET,
        market_model=OptionPricingModel.BINOMIAL,
        model_price=ModelPrice.ASK,
        load_spot=True,
        load_dividend=True,
        load_forward=True,
        load_option_spot=True,
        load_vol=True,
        load_greek=True,
        load_rates=True,
        undo_adjust=True,
        rt=True,
    )
    data_packet = _load_model_data_timeseries(request)
    rt_full_data.append(data_packet)


2026-02-13 13:07:29 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:07:29 [test] trade.datamanager.vars INFO: Timeseries for SBUX already loaded.
2026-02-13 13:07:29 [test] trade.datamanager.utils INFO: Cached data date range 2017-01-03 to 2026-02-12 does not cover requested range 2017-01-03 to 2026-02-13.
2026-02-13 13:07:29 [test] trade.datamanager.utils INFO: Cache partially covers requested date range for timeseries data structure. Key: SBUX. Fetching missing dates within range: []
2026-02-13 13:07:29 [test] trade.datamanager.utils INFO: Cutting off today's data for key: SBUX to avoid saving partial day data.
2026-02-13 13:07:30 [test] trade.datamanager.utils INFO: Caching timeseries data structure for key: SBUX with date range 2017-01-03 to 2026-02-12, missing dates within range: []
2026-02-13 13:07:30 [test] trade.datamanager.market_data INFO: Loaded spot data for symbol SBUX into cache.
2026-02-13 13:07:30 [test] trade.datamanager.utils INFO

In [ ]:
OptionSpotDataManager("SBUX").rt(
    strike=100.0,
    right="C",
    expiration="2026-09-18",
).price

datetime
2026-02-02    5.625
Name: midpoint, dtype: float64

In [ ]:
from dbase.database.db_utils import set_environment_context
set_environment_context("long_bbands")
from algo.positions.loaders.position_vars import get_position_data

2026-02-02 11:46:10 [long_bbands] algo.__init__ CRITICAL: ALGO_DIR not on main branch; skipping runtime safeguards.
[get_engine] Creating engine for DB: master_config (base: master_config), PID: 8931
[get_engine] Creating engine for DB: portfolio_data_long_bbands (base: portfolio_data), PID: 8931


Loading BokehJS ...

In [ ]:
pos = get_position_data(force=True)

2026-02-02 11:46:17 [long_bbands] algo.positions.loaders.position_vars INFO: Loading position data for today: 2026-02-02
2026-02-02 11:46:17 [test] DataManager.py CRITICAL: Skipping MySQL query. This is not optimized and may lead to performance issues.
2026-02-02 11:46:17 [long_bbands] algo.positions.loaders.position_vars INFO: Loading position data (force=True, refresh=True, eod_block=False, is_today=False, date=2026-02-02)
[get_engine] Creating engine for DB: portfolio_config_long_bbands (base: portfolio_config), PID: 8931
2026-02-02 11:46:21 [long_bbands] algo.strategies._config_utils INFO: No configuration differences found for slug 'long_bbands'.
2026-02-02 11:46:21 [long_bbands] algo.strategies._config_utils INFO: No configuration differences found for slug 'long_bbands'.
2026-02-02 11:46:22 [long_bbands] algo.strategies._config_utils INFO: No configuration differences found for slug 'long_bbands'.
Fetching rates data from yfinance directly during market hours
Fetching rates data

In [ ]:
pos.strategies["long_bbands"].positions[2].position_data.greeks

OptionGreeks(bs_delta=0.0, bs_gamma=0.0, bs_theta=0.01783721107952374, bs_vega=0.0, bs_rho=-0.01517916777565631, bs_volga=nan, binomial_delta=0.0, binomial_gamma=0.0, binomial_theta=0.01783721107952374, binomial_vega=0.0, binomial_rho=-0.01517916777565631, dollar_bs_delta=np.float64(0.0), dollar_binomial_delta=np.float64(0.0))